# Candidatos que fallan y pendientes de las búsquedas (BUG-search-short-series, PR 2b)

Un escenario por punto de la sección 12.5 (rama `fix/search-failing-candidates`), con los
filtros por defecto de Python: lo que ve un usuario en un notebook.

1. One-step-ahead: un candidato sin datos que entrenar, o sin puntos que evaluar en `levels`,
   y la ventana del forecaster que se pasa, que ya no se comprueba si no se usa.
2. Bayesian: un trial que falla.
3. El `ValueError` de `initial_train_size`.
4. El refit del mejor candidato con una serie demasiado corta.
5. Una serie con huecos, `OneStepAheadFold` y `dropna_from_series=True`.

Mismos datos que el notebook 08: ventas diarias de 2023 de dos tiendas y un producto nuevo
lanzado en septiembre (120 días).

In [1]:
import numpy as np
import pandas as pd
import optuna
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from skforecast.recursive import ForecasterRecursive, ForecasterRecursiveMultiSeries
from skforecast.model_selection import (
    TimeSeriesFold,
    OneStepAheadFold,
    backtesting_forecaster,
    grid_search_forecaster,
    bayesian_search_forecaster,
    grid_search_forecaster_multiseries,
    bayesian_search_forecaster_multiseries,
)

optuna.logging.set_verbosity(optuna.logging.WARNING)

rng = np.random.default_rng(42)
index = pd.date_range("2023-01-01", periods=365, freq="D")


def sales(n, level):
    t = np.arange(n)
    return (
        level + 10 * np.sin(2 * np.pi * t / 7)
        + 0.3 * rng.normal(0, 3, n).cumsum() + rng.normal(0, 2, n)
    )


series = {
    "store_1": pd.Series(sales(365, 100), index=index),
    "store_2": pd.Series(sales(365, 150), index=index),
    "new_product": pd.Series(sales(120, 50), index=index[-120:]),  # since 2023-09-03
}
cv = TimeSeriesFold(steps=14, initial_train_size="2023-09-30", refit=False)
cv_osa = OneStepAheadFold(initial_train_size="2023-09-30")


def multiseries(lags=7):
    return ForecasterRecursiveMultiSeries(
        Ridge(), lags=lags, transformer_series=StandardScaler()
    )


{k: (v.index[0].date(), len(v)) for k, v in series.items()}

{'store_1': (datetime.date(2023, 1, 1), 365),
 'store_2': (datetime.date(2023, 1, 1), 365),
 'new_product': (datetime.date(2023, 9, 3), 120)}

## 1. One-step-ahead: candidatos sin datos

Antes: con `OneStepAheadFold`, un `lags` sin datos que entrenar abortaba toda la búsqueda
con un `ValueError`. Ahora sus candidatos se saltan con "Parameters skipped", como con
`TimeSeriesFold`. Con 273 días hasta el final del entrenamiento, `lags=300` no tiene datos.

In [2]:
results = grid_search_forecaster_multiseries(
    forecaster    = multiseries(),
    series        = series,
    cv            = cv_osa,
    param_grid    = {"alpha": [0.1, 1, 10]},
    lags_grid     = [7, 300],
    metric        = "mean_absolute_error",
    return_best   = False,
    show_progress = False,
)
results[["lags_label", "params", "mean_absolute_error__weighted_average", "n_evaluated"]]

╭─────────────────────────── OneStepAheadValidationWarning ────────────────────────────╮
│ One-step-ahead predictions are used for faster model comparison, but they may not    │
│ fully represent multi-step prediction performance. It is recommended to backtest the │
│ final model for a more accurate multi-step performance estimate.                     │
│                                                                                      │
│ Category : skforecast.exceptions.OneStepAheadValidationWarning                       │
│ Location :                                                                           │
│ /var/folders/wt/8tvn563d5v55nspfbydgqb9r0000gp/T/ipykernel_8583/3759788513.py:1      │
│ Suppress : warnings.simplefilter('ignore', category=OneStepAheadValidationWarning)   │
╰──────────────────────────────────────────────────────────────────────────────────────╯

/var/folders/wt/8tvn563d5v55nspfbydgqb9r0000gp/T/ipykernel_8583/3759788513.py:1: RuntimeWarning: Parameters skipped: 3 of 6 candidates raise an error, so they are not in the results.

ValueError: `initial_train_size` must be greater than the `window_size` of the forecaster (300), so that the first training set has data to train with. Got '2023-09-30', 273 observations in the training set.
  - lags: [1, 2, ..., 300] (3 candidates)
  results = grid_search_forecaster_multiseries(


,lags_label,params,mean_absolute_error__weighted_average,n_evaluated
0,"[1, 2, 3, 4, 5, 6, 7]",{'alpha': 10},2.409593,276
1,"[1, 2, 3, 4, 5, 6, 7]",{'alpha': 1},2.423196,276
2,"[1, 2, 3, 4, 5, 6, 7]",{'alpha': 0.1},2.425829,276


Con `levels=["new_product"]` y `lags=35`, el producto nuevo (28 días antes de la
validación) no entra en el entrenamiento, así que no hay nada que evaluar. Antes salían filas
con `n_evaluated=0` y métricas NaN; ahora se saltan.

In [3]:
results = grid_search_forecaster_multiseries(
    forecaster    = multiseries(),
    series        = series,
    cv            = cv_osa,
    param_grid    = {"alpha": [0.1, 1]},
    lags_grid     = [7, 35],
    levels        = ["new_product"],
    metric        = "mean_absolute_error",
    return_best   = False,
    show_progress = False,
)
results[["lags_label", "params", "mean_absolute_error", "n_evaluated"]]

╭─────────────────────────── OneStepAheadValidationWarning ────────────────────────────╮
│ One-step-ahead predictions are used for faster model comparison, but they may not    │
│ fully represent multi-step prediction performance. It is recommended to backtest the │
│ final model for a more accurate multi-step performance estimate.                     │
│                                                                                      │
│ Category : skforecast.exceptions.OneStepAheadValidationWarning                       │
│ Location :                                                                           │
│ /var/folders/wt/8tvn563d5v55nspfbydgqb9r0000gp/T/ipykernel_8583/1612162856.py:1      │
│ Suppress : warnings.simplefilter('ignore', category=OneStepAheadValidationWarning)   │
╰──────────────────────────────────────────────────────────────────────────────────────╯

/var/folders/wt/8tvn563d5v55nspfbydgqb9r0000gp/T/ipykernel_8583/1612162856.py:1: RuntimeWarning: Parameters skipped: 2 of 4 candidates raise an error, so they are not in the results.

ValueError: No level of `levels` can be evaluated in the test set with this `window_size` (35). 'new_product' has 28 observations in the training set, not more than the `window_size`. Review `levels`, `initial_train_size` of `cv` or the span of the series.
  - lags: [1, 2, ..., 35] (2 candidates)
  results = grid_search_forecaster_multiseries(


,lags_label,params,mean_absolute_error,n_evaluated
0,"[1, 2, 3, 4, 5, 6, 7]",{'alpha': 1},2.640419,92
1,"[1, 2, 3, 4, 5, 6, 7]",{'alpha': 0.1},2.644516,92


La ventana del forecaster que se pasa ya no se comprueba al principio, solo la de cada
candidato. Antes, `ForecasterRecursive(lags=300)` con `lags_grid=[7, 14]` abortaba con
`OneStepAheadFold` ("`initial_train_size` must be greater than the `window_size` of the
forecaster (300)"), aunque ningún candidato usa 300 lags; con `TimeSeriesFold` funcionaba.
Ahora se evalúan los candidatos de `lags_grid` sin avisos de candidatos saltados.

In [4]:
forecaster = ForecasterRecursive(Ridge(), lags=300)
results = grid_search_forecaster(
    forecaster    = forecaster,
    y             = series["store_1"],
    cv            = cv_osa,
    param_grid    = {"alpha": [0.1, 1]},
    lags_grid     = [7, 14],
    metric        = "mean_absolute_error",
    return_best   = True,
    verbose       = False,
    show_progress = False,
)
print(f"Lags of the forecaster returned: {forecaster.max_lag}")
results[["lags_label", "params", "mean_absolute_error", "n_evaluated"]]

╭─────────────────────────── OneStepAheadValidationWarning ────────────────────────────╮
│ One-step-ahead predictions are used for faster model comparison, but they may not    │
│ fully represent multi-step prediction performance. It is recommended to backtest the │
│ final model for a more accurate multi-step performance estimate.                     │
│                                                                                      │
│ Category : skforecast.exceptions.OneStepAheadValidationWarning                       │
│ Location :                                                                           │
│ /var/folders/wt/8tvn563d5v55nspfbydgqb9r0000gp/T/ipykernel_8583/4003985839.py:2      │
│ Suppress : warnings.simplefilter('ignore', category=OneStepAheadValidationWarning)   │
╰──────────────────────────────────────────────────────────────────────────────────────╯

Lags of the forecaster returned: 14


,lags_label,params,mean_absolute_error,n_evaluated
0,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]",{'alpha': 0.1},2.243129,92
1,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]",{'alpha': 1},2.243172,92
2,"[1, 2, 3, 4, 5, 6, 7]",{'alpha': 1},2.407063,92
3,"[1, 2, 3, 4, 5, 6, 7]",{'alpha': 0.1},2.407121,92


## 2. Bayesian: un trial que falla

Antes, el primer trial que fallaba abortaba el study (y Optuna imprimía su traza). Ahora el
trial queda `FAIL` y se avisa con su número en "Parameters skipped".

In [5]:
results, study = bayesian_search_forecaster(
    forecaster    = ForecasterRecursive(Ridge(), lags=7),
    y             = series["store_1"],
    cv            = cv,
    search_space  = lambda trial: {
        "lags": trial.suggest_categorical("lags", [7, 14, 300]),
        "alpha": trial.suggest_float("alpha", 0.01, 10, log=True),
    },
    metric        = "mean_absolute_error",
    n_trials      = 10,
    return_best   = True,
    show_progress = False,
)
pd.Series([t.state.name for t in study.trials]).value_counts()

/var/folders/wt/8tvn563d5v55nspfbydgqb9r0000gp/T/ipykernel_8583/3490076669.py:1: RuntimeWarning: Parameters skipped: 3 of 10 trials raise an error, so they are not in the results.

ValueError: `initial_train_size` must be greater than the `window_size` of the forecaster (300), so that the first training set has data to train with. Got '2023-09-30', 273 observations in the training set.
  - trial 1, params: {'lags': 300, 'alpha': 1.133676686701445}
  - trial 4, params: {'lags': 300, 'alpha': 0.3939877885425336}
  - trial 9, params: {'lags': 300, 'alpha': 0.3202654041533764}
  results, study = bayesian_search_forecaster(


COMPLETE    7
FAIL        3
Name: count, dtype: int64

Un error del `search_space` (aquí, las claves del diccionario no coinciden con los nombres
de `suggest_*`) sigue abortando: se repetiría en todos los trials.

In [6]:
try:
    bayesian_search_forecaster(
        forecaster    = ForecasterRecursive(Ridge(), lags=7),
        y             = series["store_1"],
        cv            = cv,
        search_space  = lambda trial: {"ridge_alpha": trial.suggest_float("alpha", 0.01, 10)},
        metric        = "mean_absolute_error",
        n_trials      = 3,
        return_best   = False,
        show_progress = False,
    )
except ValueError as e:
    print(f"ValueError: {e}")

ValueError: `search_space` dict keys must match the names passed to `trial.suggest_*()`.
  Dict keys    : ['ridge_alpha']
  Suggest names: ['alpha']


## 3. El `ValueError` de `initial_train_size`

Antes, con una fecha y `lags=300`: "If `initial_train_size` is an integer, it must be greater
than the `window_size` of the forecaster (300) and smaller than the length of `y` (365). If it
is a date, it must be within this range of the index." Ahora dice qué límite falla y el valor
recibido.

In [7]:
try:
    backtesting_forecaster(
        forecaster    = ForecasterRecursive(Ridge(), lags=300),
        y             = series["store_1"],
        cv            = cv,
        metric        = "mean_absolute_error",
        show_progress = False,
    )
except ValueError as e:
    print(f"ValueError: {e}")

ValueError: `initial_train_size` must be greater than the `window_size` of the forecaster (300), so that the first training set has data to train with. Got '2023-09-30', 273 observations in the training set.


Con una ventana mayor que la serie ningún valor sirve, y el mensaje lo dice.

In [8]:
try:
    backtesting_forecaster(
        forecaster    = ForecasterRecursive(Ridge(), lags=400),
        y             = series["store_1"],
        cv            = cv,
        metric        = "mean_absolute_error",
        show_progress = False,
    )
except ValueError as e:
    print(f"ValueError: {e}")

ValueError: The `window_size` of the forecaster (400) leaves no valid `initial_train_size`, which must be greater than the `window_size` and smaller than the length of `y` (365). Reduce the `window_size` (lags, window features, differentiation...) or use more data.


`initial_train_size` igual a la ventana (7) no deja datos que entrenar: antes pasaba el
chequeo y fallaba después con "Length of `y` must be greater than the maximum window size
needed by the forecaster".

In [9]:
try:
    backtesting_forecaster(
        forecaster    = ForecasterRecursive(Ridge(), lags=7),
        y             = series["store_1"],
        cv            = TimeSeriesFold(steps=14, initial_train_size=7),
        metric        = "mean_absolute_error",
        show_progress = False,
    )
except ValueError as e:
    print(f"ValueError: {e}")

ValueError: `initial_train_size` must be greater than the `window_size` of the forecaster (7), so that the first training set has data to train with. Got 7.


## 4. Refit con una serie demasiado corta

Con `lags=150`, el producto nuevo (120 días) no se predice en ningún fold, pero antes el refit
del mejor candidato lo rechazaba y la búsqueda fallaba al final, tras evaluar todos los
candidatos. Ahora el refit lo deja fuera y lo avisa (también con `suppress_warnings=True`).

In [10]:
forecaster = multiseries()
results = grid_search_forecaster_multiseries(
    forecaster    = forecaster,
    series        = series,
    cv            = cv,
    param_grid    = {"alpha": [0.1, 1]},
    lags_grid     = [150],
    metric        = "mean_absolute_error",
    return_best   = True,
    show_progress = False,
)
forecaster.series_names_in_

╭──────────────────────────────── MissingValuesWarning ────────────────────────────────╮
│ Some levels are not predicted in folds where they have observed values, so their     │
│ metrics, and the aggregated metrics, do not include those folds. Levels never        │
│ predicted have NaN metrics and are left out of the aggregated metrics.               │
│                                                                                      │
│ Levels not predicted:                                                                │
│   - Too short for the `window_size` (150) in the training set: 'new_product' (never  │
│ predicted).                                                                          │
│                                                                                      │
│ Category : skforecast.exceptions.MissingValuesWarning                                │
│ Location :                                                                           │
│ /var/folders/wt/8tvn563d5v55nspfbydgqb9r0000gp/T/ipykernel_8583/1124387436.py:2      │
│ Suppress : warnings.simplefilter('ignore', category=MissingValuesWarning)            │
╰──────────────────────────────────────────────────────────────────────────────────────╯

/var/folders/wt/8tvn563d5v55nspfbydgqb9r0000gp/T/ipykernel_8583/1124387436.py:2: RuntimeWarning: 1 series is left out of the refit of the best candidate: it has no more observations than the `window_size` (150), too few to train the forecaster, and the candidate did not predict it in the search either. The forecaster returned is not trained with it (not in `series_names_in_`).

Series left out: 'new_product' (120 observations).
  results = grid_search_forecaster_multiseries(


['store_1', 'store_2']

## 5. Una serie con huecos y `dropna_from_series=True`

Con `OneStepAheadFold`, las filas del test con NaN en `y` o en sus lags se quitan, más cuanto
mayor es la ventana. El candidato con `lags=35` se evalúa sobre menos puntos y puede ganar el
ranking sin ser mejor. Antes no había aviso; ahora sale el mismo que en multiserie, con la
acción para una serie.

In [11]:
y_gaps = series["store_1"].copy()
y_gaps.iloc[[290, 310, 330, 350]] = np.nan  # 4 days without data in the validation

results = grid_search_forecaster(
    forecaster    = ForecasterRecursive(Ridge(), lags=7, dropna_from_series=True),
    y             = y_gaps,
    cv            = cv_osa,
    param_grid    = {"alpha": [0.1, 1]},
    lags_grid     = [7, 35],
    metric        = "mean_absolute_error",
    return_best   = False,
    show_progress = False,
)
results[["lags_label", "params", "mean_absolute_error", "n_evaluated"]]

╭─────────────────────────── OneStepAheadValidationWarning ────────────────────────────╮
│ One-step-ahead predictions are used for faster model comparison, but they may not    │
│ fully represent multi-step prediction performance. It is recommended to backtest the │
│ final model for a more accurate multi-step performance estimate.                     │
│                                                                                      │
│ Category : skforecast.exceptions.OneStepAheadValidationWarning                       │
│ Location :                                                                           │
│ /var/folders/wt/8tvn563d5v55nspfbydgqb9r0000gp/T/ipykernel_8583/3223775666.py:4      │
│ Suppress : warnings.simplefilter('ignore', category=OneStepAheadValidationWarning)   │
╰──────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────── MissingValuesWarning ────────────────────────────────╮
│ NaNs detected in `y_train`. They have been dropped because the target variable       │
│ cannot have NaN values. Same rows have been dropped from `X_train` to maintain       │
│ alignment. This is caused by interspersed NaNs in `y`.                               │
│                                                                                      │
│ Category : skforecast.exceptions.MissingValuesWarning                                │
│ Location :                                                                           │
│ /var/folders/wt/8tvn563d5v55nspfbydgqb9r0000gp/T/ipykernel_8583/3223775666.py:4      │
│ Suppress : warnings.simplefilter('ignore', category=MissingValuesWarning)            │
╰──────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────── MissingValuesWarning ────────────────────────────────╮
│ NaNs detected in `X_train`. They have been dropped. If you want to keep them, set    │
│ `forecaster.dropna_from_series = False`. Same rows have been removed from `y_train`  │
│ to maintain alignment. This is caused by interspersed NaNs in `y` or `exog`.         │
│                                                                                      │
│ Category : skforecast.exceptions.MissingValuesWarning                                │
│ Location :                                                                           │
│ /var/folders/wt/8tvn563d5v55nspfbydgqb9r0000gp/T/ipykernel_8583/3223775666.py:4      │
│ Suppress : warnings.simplefilter('ignore', category=MissingValuesWarning)            │
╰──────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────── MissingValuesWarning ────────────────────────────────╮
│ 2 of 4 candidates are evaluated on fewer points than the most complete one, so their │
│ metrics are not comparable. The best candidate of the ranking is not one of them.    │
│ The rows with missing values in `y` or in its lags are dropped                       │
│ (`dropna_from_series=True`), more with larger windows. To compare the candidates on  │
│ the same points, fill the gaps of `y` or remove the candidates with larger windows.  │
│ See the column `n_evaluated` of the results.                                         │
│                                                                                      │
│ Candidates with less than 95% of the 60 points of the most complete one:             │
│   - lags: [1, 2, ..., 35], 2 candidates (rows 2, 3): 17 (28.3%)                      │
│                                                                                      │
│ Category : skforecast.exceptions.MissingValuesWarning                                │
│ Location :                                                                           │
│ /var/folders/wt/8tvn563d5v55nspfbydgqb9r0000gp/T/ipykernel_8583/3223775666.py:4      │
│ Suppress : warnings.simplefilter('ignore', category=MissingValuesWarning)            │
╰──────────────────────────────────────────────────────────────────────────────────────╯

,lags_label,params,mean_absolute_error,n_evaluated
0,"[1, 2, 3, 4, 5, 6, 7]",{'alpha': 1},2.387559,60
1,"[1, 2, 3, 4, 5, 6, 7]",{'alpha': 0.1},2.387701,60
2,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",{'alpha': 1},2.404381,17
3,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",{'alpha': 0.1},2.404724,17
